# CausalAgentBench — Agent Evaluation Demo

This notebook follows patterns from the DeepLearning.AI **Evaluating AI Agents** course (Arize Phoenix / LLM-as-judge):

1. Start a local Phoenix server for automatic OpenAI tracing
2. Run the financial analyst agent across prompt versions, temperatures, and models
3. Log run metadata and LLM-as-judge groundedness scores
4. Pull traces from Phoenix
5. Compare whether prompt version affects task success rate

In [ ]:
import itertools
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import phoenix as px
from dotenv import load_dotenv

# Project root (one level up from notebooks/)
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

## 1. Start Phoenix server

Phoenix collects traces automatically once `OpenAIInstrumentor` is enabled in the agent module.

In [ ]:
session = px.launch_app()
print(f"Phoenix UI: {session.url}")

from agents.financial_analyst_agent import PROJECT_NAME, PROMPT_VERSIONS, run_agent, setup_tracing
from tracking.evaluation_logger import EvaluationLogger, fetch_generate_summary_spans, log_groundedness_annotations_to_phoenix

setup_tracing()
print(f"Tracing project: {PROJECT_NAME}")

## 2. Run agent experiments

Run the agent 10–20 times with different prompt versions, temperatures, and models.

In [ ]:
QUERY = "Compare AAPL and MSFT from 2023-01-01 to 2023-12-31."

experiment_grid = [
    {"prompt_version": pv, "temperature": temp, "model": "gpt-4o-mini"}
    for pv, temp in itertools.product(
        list(PROMPT_VERSIONS.keys()),
        [0.0, 0.2, 0.4, 0.6, 0.8],
    )
]

logger = EvaluationLogger(log_path=PROJECT_ROOT / "experiments" / "run_logs.csv")
run_records = []

for i, config in enumerate(experiment_grid, start=1):
    print(f"Run {i}/{len(experiment_grid)}: {config}")
    result = run_agent(QUERY, **config)
    record = logger.log_agent_result(result)
    run_records.append(record)

runs_df = pd.DataFrame(run_records)
runs_df.head()

## 3. Pull traces from Phoenix (SpanQuery)

Query `generate_summary` tool spans — the same pattern as Lab 3 in the course.

In [ ]:
from phoenix.client import Client

px_client = Client()
summary_spans_df = fetch_generate_summary_spans(px_client, project_name=PROJECT_NAME)
print(f"Retrieved {len(summary_spans_df)} generate_summary spans")
summary_spans_df.head()

## 4. Trace-linked LLM-as-judge evaluation

Join spans to run logs, score summary groundedness (with `suppress_tracing`), and log annotations back to Phoenix UI.

In [ ]:
trace_eval_df = log_groundedness_annotations_to_phoenix(
    px_client,
    runs_df,
    project_name=PROJECT_NAME,
)
trace_eval_df[["span_id", "summary", "llm_judge_score", "judge_explanation"]].head()

## 5. Does prompt version affect task success rate?

In [ ]:
summary = logger.summarize_by_prompt_version()
summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(summary["prompt_version"], summary["task_success_rate"])
ax.set_title("Task success rate by prompt version")
ax.set_xlabel("Prompt version")
ax.set_ylabel("Success rate")
ax.set_ylim(0, 1)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

---

## Next step: Causal Inference Analysis

> **Placeholder for DML / causal forests**

Once enough runs are logged, treat **prompt version** (or temperature, model) as the treatment and estimate its causal effect on:

- `task_success`
- `llm_judge_score`
- `cost_usd`

Planned approach:
1. Export `experiments/run_logs.csv` to `analysis/`
2. Fit double machine learning (DML) or causal forest models
3. Estimate ATE/CATE of prompt interventions while controlling for confounders (e.g., temperature, model)

```python
# TODO: causal inference pipeline
# from econml.dml import LinearDML
# treatment = runs_df["prompt_version"]
# outcome = runs_df["task_success"]
# ...
```